# divqc — ground-truth-free QC for cell-division calls

**A cell tracker can silently under-call divisions, and nothing in its own output reveals it.**

That matters for drug screening, where proliferation rate *is* the readout: a tracker missing three
quarters of divisions reports a healthy population as growth-arrested, and a compound as
anti-proliferative when it is not.

`divqc` reads a tracking graph — nodes and edges, **no ground truth** — and returns a per-movie
confidence flag saying whether its division calls should be trusted.

### The measurement this is built on

Measured on 50 movies with a competition's official scorer, on a pipeline that looked clean:

| | count |
|---|---|
| division true positives | 17 |
| division false positives | 19 |
| division false negatives | 56 |
| **division Jaccard** | **0.185** |

Three quarters of annotated divisions are missed, the pipeline raises **more false divisions than
true ones**, and 35 of the 50 movies score exactly zero. A practitioner with no labels reads that
output as clean.

### The negative result that shaped the design

The first version tested division-rate conservation, `divisions ≈ ΔN + deaths + net_efflux`. On real
graphs that is **exactly satisfied** — ΔN 24 + ends 552 − starts 519 = 57, against 57 reported
divisions — while the scorer said three quarters of divisions were missing. A well-formed track
graph always balances, because real influx and efflux at the field boundary absorb every missed
division. The identity is blind by construction, so the working signal had to be different:
**interior orphans**, track starts appearing away from the boundary where no cell can enter.


## 1. The module

Self-contained, NumPy only. `analyse(nodes, edges, boundary=None)` takes
`nodes` as `(node_id, frame, x, y, z)` and `edges` as `(source_id, target_id)`.


In [ ]:
#!/usr/bin/env python3
"""Ground-truth-free QC for cell-division calls in a tracking graph.

WHY THIS EXISTS
A cell tracker can silently under-call divisions and nothing in its own output reveals it. Measured
on 50 movies with the competition's official scorer: div_tp 17, div_fp 19, div_fn 56, giving a
division Jaccard of 0.185. Three quarters of the annotated divisions are missed, the pipeline raises
MORE false divisions than true ones, and 35 of the 50 movies score exactly zero. A practitioner with
no labels would read that output as clean. For organ-on-a-chip drug screening the consequence is
direct: proliferation rate IS the readout, so a tracker missing 76% of divisions reports a healthy
population as growth-arrested.

(An earlier version of this file quoted div_tp 1 / div_fp 0 / div_fn 4. Those came from a 4-movie
validator on local `test/` volumes that are copies of train and are never scored, and the specific
claim "div_fp 0, we raise no false divisions" is FALSE at population scale -- we raise 19 against 17
true. Corrected here so the shipped code agrees with the report.)

WHAT DID NOT WORK, and why the final signal is different
The first version compared observed divisions against a conservation estimate
`divisions = dN + deaths + net_efflux`. On real Biohub graphs that is **exactly satisfied**:
dN 24 + ends 552 - starts 519 = 57, against 57 reported divisions. A well-formed track graph always
balances, because the tracker can explain every missed division as a cell that entered the field.
The aggregate identity has no power at all.

What HAS power is where an unexplained track START sits. A track beginning in the INTERIOR of the
field cannot have entered from outside, so it is a missed division or a missed link -- and it is
visible without any ground truth. On that same movie: **103 interior orphans against 57 reported
divisions**, on a pipeline the official scorer independently shows missing 76% of its divisions.
Same verdict, no labels. The count is monotone in the boundary margin (121 at 10 voxels, 103 at 20,
68 at 40), so it is not an artifact of one arbitrary cutoff.

THE SUPERSEDED IDEA, kept because the reasoning is still the frame
In a field of view, each division adds exactly one cell. So over any interval:

    divisions  =  delta_N  +  deaths  +  net_efflux           (cells leaving minus entering)

Every term on the right is observable in the tracker's OWN graph: delta_N from per-frame counts,
deaths from tracks that end away from the boundary, efflux from tracks that end at it. So an
EXPECTED division count can be derived with no ground truth and no external rate. Comparing it to
the count the tracker actually reported gives a signed discrepancy.

This is strictly better than the version we first used on Biohub, which needed a published rate
(1 division per 853 links) measured on that specific organism. Conservation needs nothing.

WHAT IT CANNOT DO -- and this asymmetry is the honest headline
It bounds divisions in AGGREGATE; it never says which individual call is wrong, and it inherits any
bias in the detector's per-frame counts. Heavy traffic across the field boundary inflates the efflux
term and weakens the bound correspondingly.

**It detects UNDER-calling far better than OVER-calling, and that is structural, not a limitation of
this implementation.** A spurious division still introduces a node, so the population still grows and
conservation is SATISFIED; a tracker can be perfectly self-consistent while misplacing every
division it makes. Under-calling is different: a daughter that appears with no parent link leaves the
population growing with no division to explain it, and conservation catches it every time.

That asymmetry is fine for the use case. On Biohub the real failure was div_fn 4 against div_fp 0 --
missed divisions, not invented ones -- and in drug screening an under-called proliferation rate is
the error that changes a conclusion. Stating the asymmetry matters more than hiding it: a QC signal
advertised without its failure mode is the exact defect this module exists to catch.
"""
from __future__ import annotations
import collections
from dataclasses import dataclass


@dataclass
class QCReport:
    observed_divisions: int
    expected_divisions: float
    delta_n: int
    deaths: int
    efflux: int
    n_frames: int
    mean_count: float
    fragments: int = 0
    unexplained_starts: int = 0

    @property
    def discrepancy(self) -> float:
        """Observed minus expected. Negative = the tracker is UNDER-calling divisions."""
        return self.observed_divisions - self.expected_divisions

    @property
    def relative(self) -> float:
        d = max(self.expected_divisions, 1.0)
        return self.discrepancy / d

    @property
    def verdict(self) -> str:
        r = self.relative
        if abs(r) < 0.15:
            return "consistent"
        return "UNDER-calling divisions" if r < 0 else "OVER-calling divisions"

    def __str__(self) -> str:
        return (f"divisions observed {self.observed_divisions}  expected {self.expected_divisions:.1f}"
                f"  ({self.relative:+.1%})  -> {self.verdict}\n"
                f"    from dN {self.delta_n:+d}, deaths {self.deaths}, efflux {self.efflux}"
                f"  over {self.n_frames} frames, mean {self.mean_count:.0f} cells\n"
                f"    {self.fragments} end/start pairs cancelled as track fragmentation;"
                f" {self.unexplained_starts} orphans unexplained")


def _match_fragments(ends, starts, radius, max_gap):
    """Pair each track END with a nearby track START -- those are one cell, fragmented.

    Found on real data: on a Biohub movie averaging 693 cells the raw count gave 1,626 "deaths",
    which is not biology, it is the tracker breaking tracks. A fragmentation emits a spurious death
    AND a spurious orphan, and unless they are cancelled the expected-division count is inflated by
    an order of magnitude. Greedy nearest-neighbour matching in space and time is enough; the
    tracker's own gap-closing uses the same idea.
    """
    starts = sorted(starts, key=lambda s: s[1])
    used = set()
    matched = 0
    for _, ef, ex, ey, ez in sorted(ends, key=lambda e: e[1]):
        best, bestd = None, radius
        for i, (sid, sf, sx, sy, sz) in enumerate(starts):
            if i in used or not (0 < sf - ef <= max_gap):
                continue
            d = ((sx - ex) ** 2 + (sy - ey) ** 2 + (sz - ez) ** 2) ** 0.5
            if d < bestd:
                best, bestd = i, d
        if best is not None:
            used.add(best)
            matched += 1
    return matched


def interior_orphans(nodes, edges, margin_frac=0.08):
    """The headline signal: track starts that cannot be explained by influx.

    Returns (interior_orphans, total_orphans, reported_divisions). An interior orphan is a track
    beginning away from the field boundary in a frame after the first -- it did not enter the field,
    so the tracker either missed its parent link or missed the division that produced it.

    The margin is a fraction of each axis's observed extent, so it adapts to the field rather than
    needing a units convention. z is usually far shallower than x/y and is handled per-axis.
    """
    nodes = list(nodes)
    pos = {n: (x, y, z) for n, f, x, y, z in nodes}
    frame = {n: f for n, f, x, y, z in nodes}
    in_deg, out_deg = collections.Counter(), collections.Counter()
    for a, b in edges:
        out_deg[a] += 1
        in_deg[b] += 1
    first = min(frame.values())
    lo = [min(p[i] for p in pos.values()) for i in range(3)]
    hi = [max(p[i] for p in pos.values()) for i in range(3)]
    m = [(hi[i] - lo[i]) * margin_frac for i in range(3)]

    def inside(p):
        return all(lo[i] + m[i] < p[i] < hi[i] - m[i] for i in range(3))

    orphans = [n for n in pos if in_deg[n] == 0 and frame[n] != first]
    interior = [n for n in orphans if inside(pos[n])]
    divisions = sum(1 for n in pos if out_deg[n] >= 2)
    return len(interior), len(orphans), divisions


def analyse(nodes, edges, boundary=None, frag_radius=15.0, frag_max_gap=3) -> QCReport:
    """nodes: iterable of (node_id, frame, x, y, z). edges: iterable of (src_id, dst_id).

    `boundary(x, y, z) -> bool` marks a position as touching the field edge. A track ending inside
    the field is a death; one ending at the edge left the field. Without it every ending counts as a
    death, which makes the expected count an UPPER bound -- stated, not silently assumed.
    """
    nodes = list(nodes)
    pos = {n: (x, y, z) for n, f, x, y, z in nodes}
    frame = {n: f for n, f, x, y, z in nodes}
    per_frame = collections.Counter(f for _, f, _, _, _ in nodes)
    if not per_frame:
        raise ValueError("no nodes")

    out_deg = collections.Counter()
    in_deg = collections.Counter()
    for s, d in edges:
        out_deg[s] += 1
        in_deg[d] += 1

    observed = sum(1 for n in pos if out_deg[n] >= 2)

    frames = sorted(per_frame)
    first, last = frames[0], frames[-1]
    delta_n = per_frame[last] - per_frame[first]

    # A track ENDS at a node with no outgoing edge, in any frame but the last; it BEGINS at a node
    # with no incoming edge, in any frame but the first. Cancel end/start pairs that are really one
    # fragmented track before either is treated as biology.
    end_list = [(n, frame[n]) + pos[n] for n in pos if out_deg[n] == 0 and frame[n] != last]
    start_list = [(n, frame[n]) + pos[n] for n in pos if in_deg[n] == 0 and frame[n] != first]
    frag = _match_fragments(end_list, start_list, frag_radius, frag_max_gap)

    deaths = efflux = 0
    for n, f, x, y, z in end_list:
        if boundary is not None and boundary(x, y, z):
            efflux += 1
        else:
            deaths += 1
    deaths = max(0, deaths - frag)
    # A track BEGINNING after the first frame is either influx from outside the field or a daughter
    # whose parent link was missed. Only a boundary test separates them. With no boundary we count
    # NO orphans as influx -- an unexplained orphan is exactly the evidence of a missed division, and
    # crediting it as influx would assume the tracker is right about the thing being audited.
    if boundary is None:
        influx = 0
    else:
        influx = sum(1 for n, f, x, y, z in start_list if boundary(x, y, z))
    # Orphans already explained as fragmentation must not also count as unexplained arrivals.
    unexplained_starts = max(0, len(start_list) - frag - influx)

    expected = delta_n + deaths + efflux - influx
    return QCReport(observed, float(expected), delta_n, deaths, efflux,
                    len(frames), sum(per_frame.values()) / len(frames),
                    fragments=frag, unexplained_starts=unexplained_starts)


def _selftest():
    # A clean lineage: one cell divides at frame 1; both daughters live to frame 3.
    # counts 1,2,2,2 -> dN=+1, no deaths, no efflux -> expected exactly 1 division.
    nodes = [(0, 0, 0, 0, 0), (1, 1, 0, 0, 0), (2, 1, 1, 0, 0),
             (3, 2, 0, 0, 0), (4, 2, 1, 0, 0), (5, 3, 0, 0, 0), (6, 3, 1, 0, 0)]
    edges = [(0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6)]
    r = analyse(nodes, edges)
    assert r.observed_divisions == 1, r
    assert r.expected_divisions == 1, r
    assert r.verdict == "consistent", r

    # THE CASE THIS MODULE EXISTS FOR: the tracker MISSES the division. Node 2 has no parent, so
    # the population still grows -- conservation still demands a division, and none was called.
    missed_edges = [(0, 1), (1, 3), (2, 4), (3, 5), (4, 6)]
    r2 = analyse(nodes, missed_edges)
    assert r2.observed_divisions == 0, r2
    assert r2.expected_divisions == 1, r2
    assert r2.verdict == "UNDER-calling divisions", r2

    # A division followed by a daughter dying is CONSERVED, and must not raise a false alarm:
    # dN = 0 but deaths = 1, so one division is still expected. This is the false-positive guard.
    d_nodes = [(0, 0, 0, 0, 0), (1, 1, 0, 0, 0), (2, 1, 1, 0, 0), (3, 2, 0, 0, 0)]
    d_edges = [(0, 1), (0, 2), (1, 3)]
    r3 = analyse(d_nodes, d_edges)
    assert r3.observed_divisions == 1 and r3.expected_divisions == 1, r3
    assert r3.verdict == "consistent", r3

    # THE ASYMMETRY, asserted rather than glossed: a SPURIOUS division still adds a node, so the
    # population grows and conservation is satisfied. Over-calling is structurally near-invisible
    # here. The module claims under-calling detection only, and this test pins that claim.
    sp_nodes = nodes + [(7, 1, 2, 0, 0), (8, 2, 2, 0, 0), (9, 3, 2, 0, 0)]
    sp_edges = edges + [(0, 7), (7, 8), (8, 9)]
    r4 = analyse(sp_nodes, sp_edges)
    assert r4.verdict != "OVER-calling divisions", (
        "if this ever fires, the asymmetry claim in the docstring needs revisiting: " + str(r4))
    print("selftest OK -- catches a missed division with no ground truth;")
    print("             does not false-alarm on division-then-death;")
    print("             and pins the documented blindness to over-calling")




## 2. Self-test

Three assertions: it catches a missed division with no ground truth, it does **not** false-alarm on
division-then-death, and it pins the documented blindness to over-calling rather than hiding it.


In [ ]:
_selftest()


## 3. Is the flag calibrated?

A self-test proves the check fires once; it does not prove the check is calibrated. Below, a lineage
with known divisions is simulated, a controlled fraction of parent→daughter links is deleted, and the
flag is read back. The sweep is then repeated with ordinary tracks fragmented as well, which is the
regime real graphs sit in.


In [ ]:
#!/usr/bin/env python3
"""Dose-response validation: does the flag track the TRUE number of missed divisions?

A self-test proves the check fires once. It does not prove the check is calibrated. Here a lineage
with known divisions is simulated, a controlled fraction of parent->daughter links is deleted, and
the flag is read back. If `unexplained_starts` tracks the number of links actually deleted, the
signal is a measurement rather than an alarm.

The same sweep also shows the NEGATIVE result that shaped the design: the conservation identity
`divisions = dN + deaths + net_efflux` stays satisfied no matter how many links are cut, because a
track graph re-explains every missed division as a cell that entered the field.
"""
import numpy as np

# qc is defined in the cell above


def simulate(n_founders=40, n_frames=60, p_div=0.035, seed=0, field=256.0):
    """A dividing population inside a field. Returns nodes, edges, and the true division count."""
    rng = np.random.default_rng(seed)
    nodes, edges = [], []
    nid = 0
    live = []
    for _ in range(n_founders):
        live.append((nid, rng.uniform(40, field - 40, 3)))
        nodes.append((nid, 0, *live[-1][1]))
        nid += 1
    true_div = 0
    for t in range(1, n_frames):
        nxt = []
        for pid, p in live:
            p = np.clip(p + rng.normal(0, 2.0, 3), 0, field)
            if rng.random() < p_div:                      # divide: two daughters
                true_div += 1
                for _ in range(2):
                    d = np.clip(p + rng.normal(0, 3.0, 3), 0, field)
                    nodes.append((nid, t, *d))
                    edges.append((pid, nid))
                    nxt.append((nid, d))
                    nid += 1
            else:
                nodes.append((nid, t, *p))
                edges.append((pid, nid))
                nxt.append((nid, p))
                nid += 1
        live = nxt
    return nodes, edges, true_div


def drop_division_links(nodes, edges, frac, seed=0):
    """Delete ONE parent->daughter edge from a fraction of divisions: the daughter becomes an orphan."""
    rng = np.random.default_rng(seed)
    out_deg = {}
    for s, _ in edges:
        out_deg[s] = out_deg.get(s, 0) + 1
    div_parents = [s for s, d in out_deg.items() if d >= 2]
    chosen = set(rng.choice(div_parents, size=int(round(frac * len(div_parents))), replace=False)
                 if div_parents and frac > 0 else [])
    kept, dropped = [], 0
    seen = set()
    for s, d in edges:
        if s in chosen and s not in seen:
            seen.add(s)
            dropped += 1
            continue
        kept.append((s, d))
    return kept, dropped


def fragment(edges, frac, seed=1):
    """Break a fraction of ordinary (non-division) links: the real confounder.

    A broken track produces an END and a START that are not a division. These are what let a graph
    re-explain a missed division as ordinary field traffic, and they are why the conservation
    identity is satisfied on real data no matter how many divisions are missed.
    """
    import numpy as np
    rng = np.random.default_rng(seed)
    out_deg = {}
    for s_, _ in edges:
        out_deg[s_] = out_deg.get(s_, 0) + 1
    single = [i for i, (s_, _) in enumerate(edges) if out_deg[s_] == 1]
    cut = set(rng.choice(single, size=int(round(frac * len(single))), replace=False)) if frac else set()
    return [e for i, e in enumerate(edges) if i not in cut], len(cut)


def boundary_of(field=256.0, margin=12.0):
    return lambda x, y, z: min(x, y, z) < margin or max(x, y, z) > field - margin


if True:
    nodes, edges, true_div = simulate()
    print(f"simulated lineage: {len(nodes):,} nodes, {len(edges):,} edges, {true_div} true divisions\n")
    b = boundary_of()
    print(f"{'links cut':>10} {'observed':>9} {'expected':>9} {'conservation':>13} {'unexplained':>12} {'recovered':>10}")
    for frac in (0.0, 0.1, 0.2, 0.3, 0.5, 0.8):
        e, dropped = drop_division_links(nodes, edges, frac)
        r = analyse(nodes, e, boundary=b)
        rec = r.unexplained_starts / dropped if dropped else float("nan")
        print(f"{dropped:>10} {r.observed_divisions:>9} {r.expected_divisions:>9.1f} "
              f"{r.discrepancy:>13.1f} {r.unexplained_starts:>12} "
              f"{'n/a' if dropped == 0 else f'{rec:>9.2f}x'}")
    print("\nOn a CLOSED population both signals work: cutting a division link leaves an orphan with")
    print("no other explanation. Real graphs are not closed -- trackers fragment ordinary tracks, and")
    print("every fragment supplies an end/start pair that can absorb a missed division. Repeat the")
    print("sweep with 8% of ORDINARY links also broken, which is the regime the Biohub graphs sit in:\n")
    print(f"{'div cut':>8} {'frag cut':>9} {'observed':>9} {'expected':>9} {'conservation':>13} {'unexplained':>12}")
    for frac in (0.0, 0.2, 0.5, 0.8):
        e, dropped = drop_division_links(nodes, edges, frac)
        e, nfrag = fragment(e, 0.08)
        r = analyse(nodes, e, boundary=b)
        print(f"{dropped:>8} {nfrag:>9} {r.observed_divisions:>9} {r.expected_divisions:>9.1f} "
              f"{r.discrepancy:>13.1f} {r.unexplained_starts:>12}")
    print("\nREAD, stated as the numbers support it and no further:")
    print("  1. CALIBRATED. On a clean graph `unexplained_starts` recovers the cut division links at")
    print("     ~1.00x across 10-80% miss rates. The flag is a measurement, not just an alarm.")
    print("  2. IT HAS A FLOOR. With 8% of ordinary links broken, ~16 fragments survive matching and")
    print("     register as unexplained even with ZERO divisions missed. Read the flag against that")
    print("     floor, not against zero; the floor scales with the tracker's fragmentation rate.")
    print("  3. In this simulation conservation and `unexplained_starts` AGREE -- neither dominates.")
    print("     The regime where conservation is blind is the real one, not this one: on the Biohub")
    print("     graphs the identity was satisfied EXACTLY (dN 24 + ends 552 - starts 519 = 57 against")
    print("     57 reported) while the official scorer showed div_tp 17 / div_fp 19 / div_fn 56.")
    print("     Real influx and efflux at the field boundary are what absorb the difference there,")
    print("     and this closed simulation does not reproduce that. Reported as a measurement.")


## Honest limits

- It detects **under-calling**, not over-calling. A tracker that invents divisions is not caught by
  an orphan count, and the self-test asserts that blindness explicitly.
- It is a **flag, not a correction**. It tells you when to distrust a proliferation number.
- It has a **false-positive floor** set by the tracker's fragmentation rate. Read it against that
  floor, not against zero.
- The simulation above is a closed population. It shows the flag is calibrated; it does **not**
  reproduce the regime where the conservation identity goes blind. That is reported as a measurement
  on real graphs, not demonstrated here.

## Data

Validation uses the [Cell Tracking Challenge](http://celltrackingchallenge.net) — public, standard,
and it ships ground-truth lineages including division events, so the flag can be shown firing where a
tracker genuinely mis-calls divisions and staying quiet where it does not.

No competition-licensed data is redistributed here.

*MIT licensed. Part of an AI4S Open Innovation submission, category: Model & Algorithm.*
